In [1]:
from pyspark.sql import SparkSession
import hashlib
import numpy as np

In [25]:
def simhash(text, num_bits=64):
    vector = [0] * num_bits
    for token in text.split():
        h = int(hashlib.md5(token.encode()).hexdigest(), 16)
        for i in range(num_bits):
            vector[i] += 1 if (h >> i) & 1 else -1
    return sum(1 << i for i in range(num_bits) if vector[i] > 0)
    
def hamming_distance(x, y):
    diff = bin(x ^ y).count('1')
    return diff

def to_bands(fp, num_bands, band_width):
    mask = (1 << band_width) - 1
    return [(band_idx, (fp >> (band_idx * band_width)) & mask)
            for band_idx in range(num_bands)]




In [ ]:

def hamming_distance(x, y):
    diff = bin(x ^ y).count('1')
    return diff

def to_bands(fp, num_bands, band_width):
    mask = (1 << band_width) - 1
    return [(band_idx, (fp >> (band_idx * band_width)) & mask)
            for band_idx in range(num_bands)]

In [ ]:
def run(texts, num_bands=8, bit_per_band=8):
    spark = SparkSession.builder.appName("SimHash").master("local[4]").getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

    rdd = spark.sparkContext.parallelize(list(enumerate(texts)))
    finger_prints = rdd.map(lambda x: (x[0], simhash(x[1])))

    banded = finger_prints.flatMap(
        lambda x:[((b,v), x[0]) for b, v in to_bands(x[1], num_bands, bit_per_band)]
    )

    candidates = (banded
                  .groupByKey()
                  .filter(lambda x: len(list(x[1])) > 1)
                  .flatMap(lambda x: [(min(a,b), max(a,b))
                                      for i, a in enumerate(x[1])
                                      for b in list(x[1])[i+1:]])
                   .distinct())
    print("\n ___ Candidate near duplicate pairs ___")
    for pair in candidates.collect():
        print(f" Doc {pair[0]} <-> Doc {pair[1]}")
    
    spark.stop()


                                      

In [26]:
docs = [
    "the cat sat on the mat",
    "the cat sat on the mat today",   # near-duplicate of 0
    "spark is a distributed system",
    "spark is a fast distributed system",  # near-duplicate of 2
    "completely different text here",
]
run(docs)


 ___ Candidate near duplicate pairs ___
 Doc 2 <-> Doc 3
 Doc 0 <-> Doc 1
